# Consistency measures

ISO/IEC 25024 `Con-I-*` and ISO/IEC 5259-2 `Con-ML-*` measures: whether values agree with each other, within a column, a row, or against a reference distribution.

In [2]:
from datetime import date

import polars as pl

from dqmeasure import (
    DataFormatConsistency,
    DataRecordConsistency,
    DataValueDistribution,
    RiskOfDataInconsistency,
    SemanticConsistency,
)

## DataFormatConsistency
**"Does the value's shape match an admissible format?"**

`fit` learns the set of shapes seen in clean data (digits `d`, letters `a`, everything else kept literal).

In [3]:
clean = pl.DataFrame({"order_id": ["A-1023", "A-1044", "A-1091"]})
dirty = pl.DataFrame({"order_id": ["A-1023", "1023-A", "A-1091", "A1099"]})

measure = DataFormatConsistency("order_id").fit(clean)
measure.formats_

{'a-dddd'}

In [4]:
measure.predict(dirty)

order_id
f64
1.0
0.0
1.0
0.0


In [5]:
measure.score(dirty)

0.5

Two of four order IDs don't match the admissible shape `a-dddd`.

## DataRecordConsistency
**"Does the full record occur more than once?"**

Table-scoped: the row-level case of `RiskOfDataInconsistency`.

In [5]:
rows = pl.DataFrame({"user": ["ana", "bo", "ana", "dee"], "action": ["login", "login", "login", "click"]})

measure = DataRecordConsistency()
measure.predict(rows)

record
f64
1.0
0.0
1.0
0.0


In [6]:
measure.score(rows)

0.5

Row 1 and row 3 are exact duplicates; the other two rows are unique.

## RiskOfDataInconsistency
**"Does this value repeat elsewhere in the column?"**

Column-scoped. Apply it where repetition signals redundant storage (IDs), not where it's natural (categories).

In [7]:
years = pl.DataFrame({"year": [1998, 1999, 1999, 1999]})

measure = RiskOfDataInconsistency("year")
measure.predict(years)

year
f64
0.0
1.0
1.0
1.0


In [8]:
measure.score(years)

0.75

## SemanticConsistency
**"Does the record satisfy a rule relating this column to others?"**

`fit` mines single-tuple denial constraints from clean data by default (`method="dc"`). no LLM needed. `method="llm"` asks a model to propose rules instead, not executed here (needs an OpenAI API-compliant endpoint, e.g. ollama or openrouter).

In [9]:
staff = pl.DataFrame(
    {
        "born": [date(1990, 1, 1), date(1985, 6, 15), date(2000, 3, 20)],
        "recruited": [date(2010, 1, 1), date(2012, 6, 15), date(2019, 3, 20)],
    }
)
candidates = pl.DataFrame(
    {
        "born": [date(1990, 1, 1), date(1985, 6, 15), date(2004, 3, 20)],
        "recruited": [date(2010, 1, 1), date(2012, 6, 15), date(2019, 3, 20)],
    }
)  # the third candidate was recruited only 15 years after being born, younger than the clean pattern

measure = SemanticConsistency("recruited").fit(staff)
measure.rule_descriptions_

['recruited - born >= 6939 days, 0:00:00']

In [10]:
measure.predict(candidates)

recruited
f64
1.0
1.0
0.0


In [11]:
measure.score(candidates)

0.6666666666666666

In [ ]:
llm_measure = SemanticConsistency("recruited", method="llm").fit(staff)
llm_measure.rule_descriptions_

The mined rule catches the candidate recruited too soon after birth; the other two satisfy it.

## DataValueDistribution
**"How far is the observed distribution from the reference?"**

Tier-2, `score`-only, there's no per-cell value to point at. Lower is better. `0` means the distributions agree.

In [12]:
clean = pl.DataFrame({"amount": [10.0, 12.0, 11.0, 9.0, 10.5, 11.5, 10.0, 9.5]})
shifted = pl.DataFrame({"amount": [20.0, 22.0, 21.0, 19.0, 20.5]})

measure = DataValueDistribution("amount").fit(clean)
measure.score(shifted)

1.0

The shifted amounts don't overlap the reference distribution at all.